<a href="https://colab.research.google.com/github/Chanthul4054/E-Policing-An-Integrated-Spatio-Temporal-Crime-Forecasting-and-Decision-Support-System/blob/Spatio-Temporal-Crime-Prediction-System/Component_1__DPP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#What is the problem/task

#Data


In [24]:
import pandas as pd

In [2]:
#Load the drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
crime_data = pd.read_csv('/content/drive/MyDrive/DSGP/CrimeData.csv')

In [26]:
crime_data

,crime,location,date,sex,victim nationality,victim age,time,weather,Latitude,Longitude,Holiday_Name,Is_Holiday,Land_Use_Type,Unnamed: 13
0,burglary,mulgampola,31.12.2019,F,Muslim,54,8:17,Cloudy,7.280544,80.616500,Non-Holiday,False,General Urban,NaN
1,burglary,car park,04.01.2020,M,Sinhala,42,2:00,Rainy,7.283445,80.619385,Non-Holiday,False,Commercial,NaN
2,theft,panideniya bus stand,08.01.2020,F,Sinhala,20,21:01,Rainy,7.256425,80.590461,Eid al-Adha,True,Commercial,NaN
3,vehicle theft,aniwatte,10.01.2020,M,Sinhala,29,12:10,Cloudy,7.290058,80.622438,Adhi Vap Full Moon Poya Day,True,General Urban,NaN
4,robbery,dutugamunu mawatha,11.01.2020,M,Sinhala,59,2:39,Rainy,7.312344,80.645687,Non-Holiday,False,General Urban,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1600,drugs,walespark,30.07.2025,M,Sinhala,65,9:19,Rainy,7.291772,80.636707,Non-Holiday,False,Commercial,NaN
1601,drugs,old peradeniya road,16.08.2025,M,Sinhala,59,4:38,Rainy,7.280930,80.619477,Non-Holiday,False,General Urban,NaN
1602,robbery,provincial education department,13.09.2025,F,Sinhala,50,7:25,Rainy,7.290341,80.633563,Non-Holiday,False,Commercial,NaN
1603,robbery,post office,17.09.2025,F,Sinhala,61,20:12,Rainy,7.292186,80.633475,Non-Holiday,False,Commercial,NaN


In [27]:
crime_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1605 entries, 0 to 1604
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   crime               1605 non-null   object 
 1   location            1605 non-null   object 
 2   date                1605 non-null   object 
 3   sex                 1605 non-null   object 
 4   victim nationality  1605 non-null   object 
 5   victim age          1605 non-null   int64  
 6   time                1605 non-null   object 
 7   weather             1605 non-null   object 
 8   Latitude            1605 non-null   float64
 9   Longitude           1605 non-null   float64
 10  Holiday_Name        1605 non-null   object 
 11  Is_Holiday          1605 non-null   bool   
 12  Land_Use_Type       1605 non-null   object 
 13  Unnamed: 13         1 non-null      object 
dtypes: bool(1), float64(2), int64(1), object(10)
memory usage: 164.7+ KB


##Perform Spatial Join to add the divisional secreteriat.

In [28]:
import geopandas as gpd
from shapely.geometry import Point

In [29]:

# 2. Load the boundary map you just uploaded
# Note: Ensure the file name matches exactly
boundaries = gpd.read_file('/content/drive/MyDrive/DSGP/geoBoundaries-LKA-ADM3.geojson')

# 3. Convert the crime CSV into a Geospatial Dataframe
# Using the Latitude and Longitude columns from your file
geometry = [Point(xy) for xy in zip(crime_data['Longitude'], crime_data['Latitude'])]
crime_gdf = gpd.GeoDataFrame(crime_data, geometry=geometry)

# 4. Set the coordinate reference system (CRS) to match the GeoJSON (WGS84)
crime_gdf.set_crs(epsg=4326, inplace=True)
boundaries = boundaries.to_crs(epsg=4326)

# 5. Perform the Spatial Join (Point-in-Polygon)
# This finds which 'shapeName' (DS Division) contains each crime point
crime_data_update = gpd.sjoin(crime_gdf, boundaries[['shapeName', 'geometry']], how='left', predicate='within')

# 6. Rename the new column and clean up
crime_data_update = crime_data_update.rename(columns={'shapeName': 'DS_Division'})
crime_data_update = crime_data_update.drop(columns=['index_right', 'geometry'])

# Save the result
crime_data_update.to_csv('/content/drive/MyDrive/DSGP/CrimeData_Final.csv', index=False)

print("Success! 'DS_Division' column added.")
print(crime_data_update[['location', 'DS_Division']].head())

Success! 'DS_Division' column added.
               location       DS_Division
0            mulgampola  Gangawata Korale
1              car park  Gangawata Korale
2  panideniya bus stand  Gangawata Korale
3              aniwatte  Gangawata Korale
4    dutugamunu mawatha  Gangawata Korale


In [30]:
crime_data_update


,crime,location,date,sex,victim nationality,victim age,time,weather,Latitude,Longitude,Holiday_Name,Is_Holiday,Land_Use_Type,Unnamed: 13,DS_Division
0,burglary,mulgampola,31.12.2019,F,Muslim,54,8:17,Cloudy,7.280544,80.616500,Non-Holiday,False,General Urban,NaN,Gangawata Korale
1,burglary,car park,04.01.2020,M,Sinhala,42,2:00,Rainy,7.283445,80.619385,Non-Holiday,False,Commercial,NaN,Gangawata Korale
2,theft,panideniya bus stand,08.01.2020,F,Sinhala,20,21:01,Rainy,7.256425,80.590461,Eid al-Adha,True,Commercial,NaN,Gangawata Korale
3,vehicle theft,aniwatte,10.01.2020,M,Sinhala,29,12:10,Cloudy,7.290058,80.622438,Adhi Vap Full Moon Poya Day,True,General Urban,NaN,Gangawata Korale
4,robbery,dutugamunu mawatha,11.01.2020,M,Sinhala,59,2:39,Rainy,7.312344,80.645687,Non-Holiday,False,General Urban,NaN,Gangawata Korale
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1600,drugs,walespark,30.07.2025,M,Sinhala,65,9:19,Rainy,7.291772,80.636707,Non-Holiday,False,Commercial,NaN,Gangawata Korale
1601,drugs,old peradeniya road,16.08.2025,M,Sinhala,59,4:38,Rainy,7.280930,80.619477,Non-Holiday,False,General Urban,NaN,Gangawata Korale
1602,robbery,provincial education department,13.09.2025,F,Sinhala,50,7:25,Rainy,7.290341,80.633563,Non-Holiday,False,Commercial,NaN,Gangawata Korale
1603,robbery,post office,17.09.2025,F,Sinhala,61,20:12,Rainy,7.292186,80.633475,Non-Holiday,False,Commercial,NaN,Gangawata Korale


In [31]:
crime_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1605 entries, 0 to 1604
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   crime               1605 non-null   object 
 1   location            1605 non-null   object 
 2   date                1605 non-null   object 
 3   sex                 1605 non-null   object 
 4   victim nationality  1605 non-null   object 
 5   victim age          1605 non-null   int64  
 6   time                1605 non-null   object 
 7   weather             1605 non-null   object 
 8   Latitude            1605 non-null   float64
 9   Longitude           1605 non-null   float64
 10  Holiday_Name        1605 non-null   object 
 11  Is_Holiday          1605 non-null   bool   
 12  Land_Use_Type       1605 non-null   object 
 13  Unnamed: 13         1 non-null      object 
dtypes: bool(1), float64(2), int64(1), object(10)
memory usage: 164.7+ KB


##Changing data type of date and time

In [32]:
# Convert 'date' column to datetime objects
crime_data_update['date'] = pd.to_datetime(crime_data_update['date'], format='%d.%m.%Y')

# Convert 'time' column to a proper time format (Timedelta or Datetime)
crime_data_update['time'] = pd.to_datetime(crime_data_update['time'], format='%H:%M', errors='coerce').dt.time

#GETTING THE DERIVED FEATURES

In [33]:
# 2. Extract Month and Day of Week from 'date'
# We specify the format '%d.%m.%Y' to match your data (e.g., 31.12.2019)
date_converted = pd.to_datetime(crime_data_update['date'], format='%d.%m.%Y')
crime_data_update['Month'] = date_converted.dt.month
crime_data_update['Day_of_Week'] = date_converted.dt.dayofweek  # 0=Monday, 6=Sunday

# 3. Extract Hour from 'time'
# We use errors='coerce' just in case there are any badly formatted time entries
crime_data_update['Hour'] = pd.to_datetime(crime_data_update['time'], format='%H:%M', errors='coerce').dt.hour

# Save the result
crime_data_update.to_csv('/content/drive/MyDrive/DSGP/CrimeData_Final.csv', index=False)

In [34]:
crime_data_update

,crime,location,date,sex,victim nationality,victim age,time,weather,Latitude,Longitude,Holiday_Name,Is_Holiday,Land_Use_Type,Unnamed: 13,DS_Division,Month,Day_of_Week,Hour
0,burglary,mulgampola,2019-12-31,F,Muslim,54,08:17:00,Cloudy,7.280544,80.616500,Non-Holiday,False,General Urban,NaN,Gangawata Korale,12,1,NaN
1,burglary,car park,2020-01-04,M,Sinhala,42,02:00:00,Rainy,7.283445,80.619385,Non-Holiday,False,Commercial,NaN,Gangawata Korale,1,5,NaN
2,theft,panideniya bus stand,2020-01-08,F,Sinhala,20,21:01:00,Rainy,7.256425,80.590461,Eid al-Adha,True,Commercial,NaN,Gangawata Korale,1,2,NaN
3,vehicle theft,aniwatte,2020-01-10,M,Sinhala,29,12:10:00,Cloudy,7.290058,80.622438,Adhi Vap Full Moon Poya Day,True,General Urban,NaN,Gangawata Korale,1,4,NaN
4,robbery,dutugamunu mawatha,2020-01-11,M,Sinhala,59,02:39:00,Rainy,7.312344,80.645687,Non-Holiday,False,General Urban,NaN,Gangawata Korale,1,5,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1600,drugs,walespark,2025-07-30,M,Sinhala,65,09:19:00,Rainy,7.291772,80.636707,Non-Holiday,False,Commercial,NaN,Gangawata Korale,7,2,NaN
1601,drugs,old peradeniya road,2025-08-16,M,Sinhala,59,04:38:00,Rainy,7.280930,80.619477,Non-Holiday,False,General Urban,NaN,Gangawata Korale,8,5,NaN
1602,robbery,provincial education department,2025-09-13,F,Sinhala,50,07:25:00,Rainy,7.290341,80.633563,Non-Holiday,False,Commercial,NaN,Gangawata Korale,9,5,NaN
1603,robbery,post office,2025-09-17,F,Sinhala,61,20:12:00,Rainy,7.292186,80.633475,Non-Holiday,False,Commercial,NaN,Gangawata Korale,9,2,NaN


In [35]:
crime_data_update.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1605 entries, 0 to 1604
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   crime               1605 non-null   object        
 1   location            1605 non-null   object        
 2   date                1605 non-null   datetime64[ns]
 3   sex                 1605 non-null   object        
 4   victim nationality  1605 non-null   object        
 5   victim age          1605 non-null   int64         
 6   time                1605 non-null   object        
 7   weather             1605 non-null   object        
 8   Latitude            1605 non-null   float64       
 9   Longitude           1605 non-null   float64       
 10  Holiday_Name        1605 non-null   object        
 11  Is_Holiday          1605 non-null   bool          
 12  Land_Use_Type       1605 non-null   object        
 13  Unnamed: 13         1 non-null      object        
 1

In [19]:
crime_data_update.isnull()

,crime,location,date,sex,victim nationality,victim age,time,weather,Latitude,Longitude,Holiday_Name,Is_Holiday,Land_Use_Type,Unnamed: 13,DS_Division,Month,Day_of_Week,Hour
0,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1600,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
1601,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
1602,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
1603,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False


In [15]:
crime_data_update['Unnamed: 13']

,Unnamed: 13
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
1600,NaN
1601,NaN
1602,NaN
1603,NaN


##Data Cleaning

In [20]:
crime_data_update.drop(['Unnamed: 13'], inplace=True, axis=1)

In [18]:
crime_data_update

,crime,location,date,sex,victim nationality,victim age,time,weather,Latitude,Longitude,Holiday_Name,Is_Holiday,Land_Use_Type,DS_Division,Month,Day_of_Week,Hour
0,burglary,mulgampola,31.12.2019,F,Muslim,54,8:17,Cloudy,7.280544,80.616500,Non-Holiday,False,General Urban,Gangawata Korale,12,1,8
1,burglary,car park,04.01.2020,M,Sinhala,42,2:00,Rainy,7.283445,80.619385,Non-Holiday,False,Commercial,Gangawata Korale,1,5,2
2,theft,panideniya bus stand,08.01.2020,F,Sinhala,20,21:01,Rainy,7.256425,80.590461,Eid al-Adha,True,Commercial,Gangawata Korale,1,2,21
3,vehicle theft,aniwatte,10.01.2020,M,Sinhala,29,12:10,Cloudy,7.290058,80.622438,Adhi Vap Full Moon Poya Day,True,General Urban,Gangawata Korale,1,4,12
4,robbery,dutugamunu mawatha,11.01.2020,M,Sinhala,59,2:39,Rainy,7.312344,80.645687,Non-Holiday,False,General Urban,Gangawata Korale,1,5,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1600,drugs,walespark,30.07.2025,M,Sinhala,65,9:19,Rainy,7.291772,80.636707,Non-Holiday,False,Commercial,Gangawata Korale,7,2,9
1601,drugs,old peradeniya road,16.08.2025,M,Sinhala,59,4:38,Rainy,7.280930,80.619477,Non-Holiday,False,General Urban,Gangawata Korale,8,5,4
1602,robbery,provincial education department,13.09.2025,F,Sinhala,50,7:25,Rainy,7.290341,80.633563,Non-Holiday,False,Commercial,Gangawata Korale,9,5,7
1603,robbery,post office,17.09.2025,F,Sinhala,61,20:12,Rainy,7.292186,80.633475,Non-Holiday,False,Commercial,Gangawata Korale,9,2,20


In [21]:
crime_data_update.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1605 entries, 0 to 1604
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   crime               1605 non-null   object 
 1   location            1605 non-null   object 
 2   date                1605 non-null   object 
 3   sex                 1605 non-null   object 
 4   victim nationality  1605 non-null   object 
 5   victim age          1605 non-null   int64  
 6   time                1605 non-null   object 
 7   weather             1605 non-null   object 
 8   Latitude            1605 non-null   float64
 9   Longitude           1605 non-null   float64
 10  Holiday_Name        1605 non-null   object 
 11  Is_Holiday          1605 non-null   bool   
 12  Land_Use_Type       1605 non-null   object 
 13  DS_Division         1605 non-null   object 
 14  Month               1605 non-null   int32  
 15  Day_of_Week         1605 non-null   int32  
 16  Hour       

In [23]:
print(type(crime_data_update['date']))

<class 'pandas.core.series.Series'>
